# Lesson 1.2 — Action space and control modes

An action is never fully described by its tensor. This notebook makes that
concrete by varying one argument — `control_mode` — on the **same task** and
watching the action space change underneath.

The lesson's Action Specification is:

```text
(space, representation, frame, dimension, semantics, unit, frequency, controller mode)
```

Every row of the comparison table below fills in a different subset of that tuple.
Matching dimensions do **not** imply matching semantics.

## 1.2.1 — The shared preamble

In [1]:
import gymnasium as gym
import mani_skill.envs
import numpy as np
import torch

torch.set_printoptions(precision=4, sci_mode=False)


def make_env(obs_mode="state", control_mode="pd_joint_delta_pos", seed=0):
    env = gym.make(
        "PickCube-v1",
        obs_mode=obs_mode,
        control_mode=control_mode,
        num_envs=1,
    )
    env.reset(seed=seed)
    return env

## 1.2.2 — Enumerate every control mode

Same `PickCube-v1` task, eleven control modes, and action dimensions ranging from
**4 to 15**. Recognising this range is the point: "an 8-dimensional action" means
nothing on its own.

In [2]:
agent = make_env().unwrapped.agent
modes = sorted(agent.supported_control_modes)

print(f"{'control mode':<28} {'action space':<52} dim")
print("-" * 92)
for mode in modes:
    try:
        e = make_env(control_mode=mode)
        space = e.action_space
        shape = space.shape
        if np.allclose(space.low, -1.0) and np.allclose(space.high, 1.0):
            desc = f"Box(-1, 1, {shape})  normalized"
        else:
            desc = f"Box(lo, hi, {shape})  physical limits"
        print(f"{mode:<28} {desc:<52} {shape[0]}")
        e.close()
    except Exception as exc:
        print(f"{mode:<28} FAILED: {type(exc).__name__}: {exc}")

2026-09-22 11:14:12,657 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


2026-09-22 11:14:13,012 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


2026-09-22 11:14:13,159 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


control mode                 action space                                         dim
--------------------------------------------------------------------------------------------
pd_ee_delta_pos              Box(-1, 1, (4,))  normalized                         4


...

2026-09-22 11:14:14,184 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


2026-09-22 11:14:14,321 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


pd_joint_pos                 Box(lo, hi, (8,))  physical limits                   8
pd_joint_pos_vel             Box(lo, hi, (15,))  physical limits                  15


2026-09-22 11:14:14,451 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


pd_joint_target_delta_pos    Box(-1, 1, (8,))  normalized                         8
pd_joint_vel                 Box(-1, 1, (8,))  normalized                         8


### Why the dimensions differ

| Control mode | Dim | Semantics |
|---|---:|---|
| `pd_joint_delta_pos` | 8 | 7 joint deltas + 1 gripper |
| `pd_joint_pos` | 8 | 7 absolute joint targets + 1 gripper, **physical limits** not `[-1,1]` |
| `pd_joint_vel` | 8 | 7 joint velocities + gripper |
| `pd_joint_pos_vel` | 15 | 8 position + 7 velocity |
| `pd_ee_delta_pos` | 4 | 3 Cartesian deltas + gripper |
| `pd_ee_delta_pose` | 7 | 6-DoF Cartesian delta + gripper |
| `pd_ee_pose` | 7 | absolute end-effector pose target |

Two facts fall out of this table:

- **joint space versus Cartesian space**: `pd_joint_*` targets joint angles,
  `pd_ee_*` targets the end-effector. The same physical motion is a different
  action in each.
- **delta versus absolute**: `pd_joint_delta_pos` and `pd_joint_pos` both have
  dimension 8, but one is a displacement and the other is a position. This is the
  clearest counterexample to "same shape means same action".

## 1.2.3 — The project's control mode: `pd_joint_delta_pos`

The dataset in this repository was collected with `pd_joint_delta_pos`. Its eight
channels are **two different semantics concatenated**:

| Channel | Controller | Semantics | Physical range |
|---|---|---|---|
| `a[0:7]` | `PDJointPosController` | normalized **delta** from the current joint position | `[-0.1, 0.1]` rad |
| `a[7]` | `PDJointPosMimicController` | normalized **absolute** gripper target | `[-0.01, 0.04]` m |

Read the configs straight from the live controller rather than trusting names.

In [3]:
env = make_env()
controller = env.unwrapped.agent.controller

print("action_mapping:", controller.action_mapping)
print("action space   :", controller.action_space)

for name, config in controller.configs.items():
    print(f"\n--- {name} ({type(config).__name__}) ---")
    for field in ("joint_names", "lower", "upper", "use_delta", "use_target",
                  "normalize_action", "mimic", "stiffness", "damping"):
        print(f"    {field:<18} {getattr(config, field, '<absent>')}")

2026-09-22 11:14:14,583 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


action_mapping: {'arm': (0, 7), 'gripper': (7, 8)}
action space   : Box(-1.0, 1.0, (8,), float32)

--- arm (PDJointPosControllerConfig) ---
    joint_names        ['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']
    lower              -0.1
    upper              0.1
    use_delta          True
    use_target         False
    normalize_action   True
    mimic              <absent>
    stiffness          1000.0
    damping            100.0

--- gripper (PDJointPosMimicControllerConfig) ---
    joint_names        ['panda_finger_joint1', 'panda_finger_joint2']
    lower              -0.01
    upper              0.04
    use_delta          False
    use_target         False
    normalize_action   True
    mimic              {'panda_finger_joint2': {'joint': 'panda_finger_joint1'}}
    stiffness          1000.0
    damping            100.0


Read those flags carefully, because together they decide what an action
*means*:

- `use_delta=True` on the arm — the action is a displacement.
- `use_target=False` — the delta is measured against the **current** joint
  position, not against the previous commanded target.
- `normalize_action=True` — the `[-1,1]` input is scaled into `[lower, upper]`.
- For the gripper, `use_delta=False` — it is an absolute position target, even
  though it lives in the same normalized `[-1,1]` box.

## 1.2.4 — Verify the arm mapping empirically

Two numbers matter, and they are different:

1. the **control target** after one step — this should move by exactly `0.1 · a`;
2. the **actual joint position** after one step — this moves much less, because the
   PD controller tracks the target over several steps.

Confusing the two is an easy way to misread action magnitudes.

In [4]:
env = make_env()
robot = env.unwrapped.agent.robot


def finger_indices(robot):
    names = [joint.name for joint in robot.active_joints]
    return names.index("panda_finger_joint1"), names.index("panda_finger_joint2")


for value in (1.0, -1.0):
    env.reset(seed=0)
    qpos_before = robot.get_qpos().clone()[0]

    action = torch.zeros(1, 8)
    action[0, :7] = value
    env.step(action)

    target = robot.get_drive_targets()
    target = target[0] if target.ndim > 1 else target

    print(f"a_arm = {value:+.1f}")
    print(f"   command implied delta : {0.1 * value:+.4f} rad")
    print(f"   drive_target - qpos   : {target[:3].cpu().numpy()}  <-- exact")
    print(f"   qpos_after - qpos     : {(robot.get_qpos()[0] - qpos_before)[:3].cpu().numpy()}  <-- partial (PD tracking)")
    print()

2026-09-22 11:14:14,661 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


a_arm = +1.0
   command implied delta : +0.1000 rad
   drive_target - qpos   : [0.13528106 0.50070226 0.11957476]  <-- exact
   qpos_after - qpos     : [0.0131324  0.03490701 0.02570832]  <-- partial (PD tracking)

a_arm = -1.0
   command implied delta : -0.1000 rad
   drive_target - qpos   : [-0.06471895  0.30070224 -0.08042524]  <-- exact
   qpos_after - qpos     : [-0.01323517 -0.03481296 -0.02594179]  <-- partial (PD tracking)



## 1.2.5 — Verify the gripper mapping, including direction

The gripper is absolute, so its target should be a fixed value per action,
independent of the current state. And the direction question — does `+1` open or
close? — is answerable by measurement instead of by guessing from the sign.

In [5]:
env = make_env()
robot = env.unwrapped.agent.robot
f1, f2 = finger_indices(robot)

for value in (-1.0, 0.0, 1.0):
    env.reset(seed=0)
    qpos_before = robot.get_qpos().clone()[0]

    action = torch.zeros(1, 8)
    action[0, 7] = value
    env.step(action)

    target = robot.get_drive_targets()
    target = target[0] if target.ndim > 1 else target

    print(f"a_gripper = {value:+.1f} -> target finger1 = {target[f1].item():+.5f}, "
          f"finger2 = {target[f2].item():+.5f}   (reset qpos was {qpos_before[f1].item():+.4f})")

print("\nBoth fingers receive the same target because finger2 mimics finger1.")
print("The gripper joint starts at its maximum value, so +1 is the open extreme.")

2026-09-22 11:14:14,758 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


a_gripper = -1.0 -> target finger1 = -0.01000, finger2 = -0.01000   (reset qpos was +0.0400)
a_gripper = +0.0 -> target finger1 = +0.01500, finger2 = +0.01500   (reset qpos was +0.0400)


a_gripper = +1.0 -> target finger1 = +0.04000, finger2 = +0.04000   (reset qpos was +0.0400)

Both fingers receive the same target because finger2 mimics finger1.
The gripper joint starts at its maximum value, so +1 is the open extreme.


## 1.2.6 — The Action Specification table

Fill this in for the project dataset. Every field is required before two datasets
can be compared:

| Field | Value |
|---|---|
| space | `Box(-1, 1, (8,), float32)` |
| representation | joint-space, mixed delta/absolute |
| coordinate frame | joint space (no Cartesian frame involved for channels 0–6) |
| dimension | 8 = 7 arm + 1 gripper |
| semantics | arm: joint position **delta**; gripper: **absolute** position target |
| unit | rad (arm), m (gripper joint position) |
| frequency | 20 Hz control (see `2.3_time_alignment.ipynb`) |
| controller mode | `pd_joint_delta_pos` |

Note the last row. This notebook establishes the semantics; the **frequency** is a
separate property and it is currently inconsistent between the dataset metadata
and the environment. That is the subject of the time-alignment notebook.

## Takeaways

1. One task argument changes the action dimension from 4 to 15. Shape alone is
   never a specification.
2. `pd_joint_delta_pos` has **two semantics in one vector**: joint deltas for the
   arm, an absolute target for the gripper.
3. `use_target=False` means the delta is applied to the current joint position, so
   the same action does not produce the same absolute motion from different states.
4. The commanded delta and the realized motion differ within a step because a PD
   controller tracks its target over time.
5. Two datasets with identical action shapes can still be mutually unusable — for
   example `pd_joint_delta_pos` versus `pd_joint_pos`, both `(8,)`.